In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import torch
import warnings
warnings.filterwarnings("ignore")

# 导入 TabNet
from pytorch_tabnet.tab_model import TabNetRegressor

# 1. 加载数据
print("正在加载数据...")
# 使用与LightGBM相同的平滑数据集
df_original = pd.read_excel('../../data/pinghua_data_x_names.xlsx')
print(f"原始数据集大小: {df_original.shape}")

# 2. 定义目标变量 - 与LightGBM保持一致
target_vars = ['x25', 'x26', 'x27', 'x28', 'x29']

# 3. 分离特征和目标
X_original = df_original.drop(target_vars, axis=1)
y_original = df_original[target_vars]

# 4. 分割原始数据为训练集和测试集
X_train, X_test, y_train, y_test = train_test_split(
    X_original, y_original, test_size=0.2, random_state=42
)

print(f"训练集大小: {X_train.shape}")
print(f"测试集大小: {X_test.shape}")

# 5. 数据标准化
scaler_X = StandardScaler()
scaler_y = StandardScaler()

X_train_scaled = scaler_X.fit_transform(X_train)
X_test_scaled = scaler_X.transform(X_test)
y_train_scaled = scaler_y.fit_transform(y_train)

# 6. 检查是否有 GPU 可用
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"使用设备: {device}")
# 最优参数:
# n_d: 22
# n_a: 128
# n_steps: 3
# gamma: 2.0
# lambda_sparse: 0.0005467072969055289
# lr: 0.029384293039604333
# batch_size: 115
# virtual_batch_size: 119
# 7. 创建TabNet模型
print("\n开始训练TabNet模型...")
tabnet_model = TabNetRegressor(
    n_d=22,                      # 决策层维度
    n_a=128,                     # 注意力层维度
    n_steps=3,                   # 步数
    gamma=2.0,                   # 稀疏度
    lambda_sparse=0.0005467072969055289,  # 额外的稀疏度
    optimizer_fn=torch.optim.Adam,
    optimizer_params=dict(lr=0.029384293039604333),  # 学习率
    scheduler_params={"step_size":50, "gamma":0.8},
    scheduler_fn=torch.optim.lr_scheduler.StepLR,
    mask_type='entmax',
    device_name=device,
    verbose=0,   # 显示训练进度
    seed=42
)

# 8. 训练TabNet模型
tabnet_model.fit(
    X_train=X_train_scaled,  # 训练集特征
    y_train=y_train_scaled,  # 训练集多输出目标
    eval_set=[(X_test_scaled, scaler_y.transform(y_test))],  # 验证集
    eval_name=['val'],
    eval_metric=['rmse'],
    max_epochs=200,             # 最大训练轮数
    patience=20,                # 提前停止的耐心值
    batch_size=227,             # 批量大小
    virtual_batch_size=96,      # 虚拟批量大小
    num_workers=0               # 数据加载器工作线程数
)
print("TabNet模型训练完成！")

# 9. 在测试集上进行预测
y_pred_scaled = tabnet_model.predict(X_test_scaled)

# 10. 将预测结果反标准化回原始空间
y_pred = scaler_y.inverse_transform(y_pred_scaled)

# 11. 评估模型性能
print("\n模型评估指标:")
overall_r2 = 0
overall_rmse = 0
overall_mae = 0

for idx, name in enumerate(target_vars):
    # 计算指标
    mse = mean_squared_error(y_test.iloc[:, idx], y_pred[:, idx])
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_test.iloc[:, idx], y_pred[:, idx])
    r2 = r2_score(y_test.iloc[:, idx], y_pred[:, idx])
    
    # 累加指标值
    overall_r2 += r2
    overall_rmse += rmse
    overall_mae += mae
    
    # 打印每个目标变量的指标
    print(f"\n{name} 指标:")
    print(f"MSE={mse:.4f}, RMSE={rmse:.4f}, MAE={mae:.4f}, R²={r2:.4f}")

# 计算平均指标
overall_r2 /= len(target_vars)
overall_rmse /= len(target_vars)
overall_mae /= len(target_vars)

print("\n总体平均指标:")
print(f"平均 RMSE: {overall_rmse:.4f}")
print(f"平均 MAE: {overall_mae:.4f}")
print(f"平均 R²: {overall_r2:.4f}")